# Ukázkový Jupyter notebook pro Astrozor

Tento notebook demonstruje **Python + matplotlib + markdown** workflow pro Astrozor. Stáhni si zdroj, spusť všechny buňky (`Cell → Run All`), vyexportuj přes:

```bash
jupyter nbconvert --to html --embed-images jupyter-notebook.ipynb
mkdir -p bundle
cp jupyter-notebook.html bundle/index.html
```

Pak publikuj `bundle/` složku přes VS Code (`Astrozor: Publish folder`) nebo curl podle [Publikování z Jupyter](/?from=docs&d=publish-jupyter).

## 1) Načtení dat o jasných hvězdách

Syntetický dataset — magnitudy a vzdálenosti známých hvězd.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dark theme pro matplotlib — splývá s Astrozor designem
plt.style.use('dark_background')

stars = {
    'name':            ['Sirius A', 'Vega', 'Polaris', 'Betelgeuse',
                        'Alfa Centauri A', 'Procyon', 'Capella', 'Aldebaran'],
    'magnitude':       [-1.46, 0.03, 1.98, 0.42, -0.27, 0.34, 0.08, 0.86],
    'distance_pc':     [2.64, 7.68, 132, 168, 1.34, 3.51, 13.2, 20.4],
    'spectral':        ['A1V', 'A0V', 'F7Ib', 'M1Ia', 'G2V', 'F5IV', 'G5III', 'K5III']
}

# Absolutní magnituda M = m - 5 log10(d/10)
absolute = [m - 5 * np.log10(d / 10) for m, d in zip(stars['magnitude'], stars['distance_pc'])]
stars['absolute'] = absolute

print(f"Načteno {len(stars['name'])} hvězd")
print(f"Rozsah absolutních magnitud: {min(absolute):.2f} až {max(absolute):.2f}")

## 2) HR-like diagram

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor('#0f172a')
ax.set_facecolor('#0f172a')

scatter = ax.scatter(
    stars['distance_pc'], stars['absolute'],
    s=120, alpha=0.85, c=stars['magnitude'], cmap='plasma',
    edgecolors='white', linewidths=0.6
)

for name, x, y in zip(stars['name'], stars['distance_pc'], stars['absolute']):
    ax.annotate(name, (x, y), xytext=(8, 6), textcoords='offset points',
                fontsize=9, color='#e2e8f0', alpha=0.9)

ax.set_xscale('log')
ax.invert_yaxis()
ax.set_xlabel('Vzdálenost (pc, log scale)', color='#e2e8f0')
ax.set_ylabel('Absolutní magnituda', color='#e2e8f0')
ax.set_title('Jasné hvězdy: vzdálenost vs. absolutní jasnost',
             color='#e2e8f0', fontsize=13)

cbar = plt.colorbar(scatter, ax=ax, label='Zdánlivá magnituda')
cbar.ax.yaxis.label.set_color('#e2e8f0')
cbar.ax.tick_params(colors='#e2e8f0')

ax.grid(True, alpha=0.2, color='#334155')
ax.tick_params(colors='#cbd5e1')
for spine in ax.spines.values():
    spine.set_color('#334155')

plt.tight_layout()
plt.show()

## 3) Pogsonův vzorec

Vztah pro rozdíl magnitud z poměru toků:

$$
m_1 - m_2 = -2.5 \log_{10}\left(\frac{F_1}{F_2}\right)
$$

Příklady: rozdíl 5 magnitud = 100× větší tok; rozdíl 1 magnituda ≈ 2.512× tok.

In [ ]:
def pogson(flux_ratio: float) -> float:
    """Vrátí rozdíl magnitud z poměru toků."""
    return -2.5 * np.log10(flux_ratio)

examples = [
    (100, '100× větší tok'),
    (10, '10× větší tok'),
    (2.512, 'klasický "jedna magnituda"'),
    (1, 'stejný tok'),
    (0.01, '100× menší tok'),
]
for ratio, label in examples:
    diff = pogson(ratio)
    print(f'F1/F2 = {ratio:>8.4f}  →  m1 - m2 = {diff:>+6.2f}  ({label})')

## 4) Závěr

Pokud tento notebook vidíš na Astrozoru jako iframe s matplotlib grafem a markdown textem — Jupyter publish pipeline funguje.

**Pro interaktivitu** (hover, zoom) místo matplotlib použij **plotly**:

```python
import plotly.express as px
fig = px.scatter(x=stars['distance_pc'], y=stars['absolute'],
                 hover_name=stars['name'])
fig.show()
```

Plotly se uloží do HTML jako interaktivní widget — funguje i bez Python kernelu po publikaci.